# ETL y Análisis de Datos con PySpark
## Proyecto: Análisis de Sentimientos de Noticias

Este notebook realiza un proceso completo de ETL (Extract, Transform, Load) utilizando PySpark para analizar un dataset de sentimientos de noticias. Incluye:
- Instalación de librerías necesarias
- Carga y exploración de datos
- Limpieza de datos (sin eliminar columnas)
- Transformaciones usando PySpark
- Visualizaciones
- Guardado del dataset limpio

**Dataset:** `stock_senti_analysis.csv` - Contiene fechas, etiquetas de sentimiento y múltiples titulares de noticias.

## 1. Instalación de Librerías Requeridas

Primero instalaremos todas las librerías necesarias para nuestro análisis con PySpark.

In [1]:
# Instalación de librerías necesarias
import subprocess
import sys

def install_package(package):
    """Función para instalar paquetes si no están disponibles"""
    try:
        __import__(package)
        print(f"✓ {package} ya está instalado")
    except ImportError:
        print(f"Instalando {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✓ {package} instalado correctamente")

# Lista de paquetes necesarios
packages = [
    "pyspark",
    "matplotlib",
    "seaborn", 
    "pandas",
    "numpy",
    "plotly",
    "findspark"
]

print("=== Instalación de Librerías ===")
for package in packages:
    install_package(package)

print("\n¡Todas las librerías han sido instaladas correctamente!")

=== Instalación de Librerías ===
Instalando pyspark...
✓ pyspark instalado correctamente
✓ matplotlib ya está instalado
✓ seaborn ya está instalado
✓ pandas ya está instalado
✓ numpy ya está instalado
✓ plotly ya está instalado
✓ findspark ya está instalado

¡Todas las librerías han sido instaladas correctamente!


## 2. Importación de Librerías e Inicialización de Spark Session

Ahora importamos todas las librerías e inicializamos la sesión de Spark para comenzar el procesamiento de datos.

In [2]:
# Importación de librerías
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Configuración de matplotlib para mejor visualización
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("=== Inicializando Spark Session ===")

# Crear Spark Session con configuración optimizada
spark = SparkSession.builder \
    .appName("ETL_Sentiment_Analysis") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

print(f"✓ Spark Session creada exitosamente!")
print(f"✓ Spark Version: {spark.version}")
print(f"✓ Spark UI disponible en: {spark.sparkContext.uiWebUrl}")

# Configurar nivel de log para reducir verbosidad
spark.sparkContext.setLogLevel("WARN")

ValueError: Couldn't find Spark, make sure SPARK_HOME env is set or Spark is in an expected location (e.g. from homebrew installation).

## 3. Carga del Dataset

Cargaremos el dataset de análisis de sentimientos usando PySpark DataFrame con las opciones apropiadas.

In [ ]:
# Definir rutas de los archivos
file_path_original = "Files/stock_senti_analysis.csv"
file_path_clean = "Files/stock_senti_analysis_limpio.csv"

print("=== Cargando Datasets ===")

# Cargar dataset original
try:
    df_original = spark.read.csv(file_path_original, 
                                header=True, 
                                inferSchema=True, 
                                multiline=True,
                                escape='"')
    print(f"✓ Dataset original cargado exitosamente desde: {file_path_original}")
    print(f"  - Número de filas: {df_original.count()}")
    print(f"  - Número de columnas: {len(df_original.columns)}")
except Exception as e:
    print(f"✗ Error cargando dataset original: {e}")

# Cargar dataset limpio (si existe)
try:
    df_clean = spark.read.csv(file_path_clean, 
                             header=True, 
                             inferSchema=True, 
                             multiline=True,
                             escape='"')
    print(f"✓ Dataset limpio cargado exitosamente desde: {file_path_clean}")
    print(f"  - Número de filas: {df_clean.count()}")
    print(f"  - Número de columnas: {len(df_clean.columns)}")
    
    # Usar dataset limpio como principal para el análisis
    df = df_clean
    print("➤ Usando dataset limpio para el análisis")
    
except Exception as e:
    print(f"⚠ Dataset limpio no encontrado, usando dataset original: {e}")
    df = df_original
    print("➤ Usando dataset original para el análisis")

print(f"\n=== Dataset Principal ===")
print(f"Filas: {df.count()}, Columnas: {len(df.columns)}")

## 4. Exploración de Datos y Análisis del Schema

Examinaremos la estructura del dataset, tipos de datos y identificaremos columnas con valores faltantes o vacíos.

In [ ]:
print("=== ANÁLISIS EXPLORATORIO DE DATOS ===\n")

# 1. Mostrar schema del DataFrame
print("1. SCHEMA DEL DATASET:")
df.printSchema()

print("\n" + "="*50)

# 2. Mostrar columnas
print("2. COLUMNAS DEL DATASET:")
columns = df.columns
for i, col in enumerate(columns):
    print(f"{i+1:2d}. {col}")

print(f"\nTotal de columnas: {len(columns)}")

print("\n" + "="*50)

# 3. Primeras filas del dataset
print("3. PRIMERAS 5 FILAS:")
df.show(5, truncate=False)

print("\n" + "="*50)

# 4. Estadísticas básicas
print("4. INFORMACIÓN BÁSICA:")
print(f"Número total de filas: {df.count()}")
print(f"Número total de columnas: {len(df.columns)}")

# 5. Análisis de tipos de datos
print("\n5. TIPOS DE DATOS:")
data_types = df.dtypes
for column, dtype in data_types:
    print(f"{column:15s} -> {dtype}")

print("\n" + "="*50)

In [ ]:
# Análisis detallado de valores nulos y vacíos
print("6. ANÁLISIS DE VALORES NULOS Y VACÍOS:")

# Función para contar valores nulos, vacíos y espacios en blanco
def analyze_missing_values(df):
    missing_data = []
    
    for column in df.columns:
        # Contar nulls
        null_count = df.filter(col(column).isNull()).count()
        
        # Contar strings vacíos
        empty_string_count = df.filter((col(column) == "")).count()
        
        # Contar espacios en blanco
        whitespace_count = df.filter(col(column).rlike("^\\s*$")).count()
        
        # Total de valores problemáticos
        total_missing = null_count + empty_string_count + whitespace_count
        
        # Porcentaje
        total_rows = df.count()
        missing_percentage = (total_missing / total_rows) * 100 if total_rows > 0 else 0
        
        missing_data.append({
            'Column': column,
            'Null_Count': null_count,
            'Empty_String': empty_string_count,
            'Whitespace': whitespace_count,
            'Total_Missing': total_missing,
            'Missing_Percentage': round(missing_percentage, 2)
        })
    
    return missing_data

# Ejecutar análisis
missing_analysis = analyze_missing_values(df)

# Mostrar resultados
print(f"{'Column':<20} {'Nulls':<8} {'Empty':<8} {'Spaces':<8} {'Total':<8} {'%':<8}")
print("-" * 70)

for item in missing_analysis:
    if item['Total_Missing'] > 0:  # Solo mostrar columnas con problemas
        print(f"{item['Column']:<20} {item['Null_Count']:<8} {item['Empty_String']:<8} "
              f"{item['Whitespace']:<8} {item['Total_Missing']:<8} {item['Missing_Percentage']:<8}")

# Contar columnas con problemas
problematic_columns = [item for item in missing_analysis if item['Total_Missing'] > 0]
print(f"\nColumnas con valores problemáticos: {len(problematic_columns)}")
print(f"Total de registros: {df.count()}")

## 5. Limpieza de Datos - Manejo de Valores Vacíos

Limpiaremos los valores vacíos usando funciones de PySpark como fillna(), dropna() y transformaciones personalizadas **sin eliminar columnas completas**.

In [ ]:
print("=== PROCESO DE LIMPIEZA DE DATOS ===\n")

# Crear copia del DataFrame para limpieza
df_cleaned = df

print("1. ESTADO INICIAL:")
print(f"Filas antes de limpieza: {df_cleaned.count()}")

# 1. Limpiar valores nulos y vacíos en columnas de texto (Top1-Top25)
print("\n2. LIMPIANDO COLUMNAS DE TITULARES (Top1-Top25):")

# Identificar columnas Top
top_columns = [col for col in df_cleaned.columns if col.startswith('Top')]
print(f"Columnas de titulares encontradas: {len(top_columns)}")

# Reemplazar valores nulos y vacíos en columnas Top con "Sin titular"
for column in top_columns:
    df_cleaned = df_cleaned.withColumn(
        column,
        when(col(column).isNull() | (col(column) == "") | col(column).rlike("^\\s*$"), 
             "Sin titular")
        .otherwise(col(column))
    )

print("✓ Valores nulos/vacíos en columnas de titulares reemplazados con 'Sin titular'")

# 2. Limpiar columna Date
print("\n3. LIMPIANDO COLUMNA DATE:")
df_cleaned = df_cleaned.withColumn(
    "Date",
    when(col("Date").isNull() | (col("Date") == ""), "1900-01-01")
    .otherwise(col("Date"))
)
print("✓ Valores nulos en Date reemplazados con '1900-01-01'")

# 3. Limpiar columna Label (importante para análisis)
print("\n4. LIMPIANDO COLUMNA LABEL:")
# Contar valores nulos en Label antes
label_nulls_before = df_cleaned.filter(col("Label").isNull()).count()
print(f"Valores nulos en Label antes: {label_nulls_before}")

# Reemplazar nulls en Label con 0 (sentimiento neutral/negativo por defecto)
df_cleaned = df_cleaned.withColumn(
    "Label",
    when(col("Label").isNull(), 0)
    .otherwise(col("Label"))
)

label_nulls_after = df_cleaned.filter(col("Label").isNull()).count()
print(f"Valores nulos en Label después: {label_nulls_after}")
print("✓ Valores nulos en Label reemplazados con 0")

# 4. Manejo de columnas adicionales (si existen en el dataset limpio)
additional_columns = [col for col in df_cleaned.columns 
                     if col not in ['Date', 'Label'] + top_columns]

if additional_columns:
    print(f"\n5. LIMPIANDO COLUMNAS ADICIONALES: {additional_columns}")
    for column in additional_columns:
        column_type = dict(df_cleaned.dtypes)[column]
        
        if column_type in ['string', 'varchar']:
            # Para columnas de texto
            df_cleaned = df_cleaned.withColumn(
                column,
                when(col(column).isNull() | (col(column) == "") | col(column).rlike("^\\s*$"), 
                     "Sin información")
                .otherwise(col(column))
            )
        elif column_type in ['int', 'integer', 'long', 'bigint']:
            # Para columnas numéricas enteras
            df_cleaned = df_cleaned.withColumn(
                column,
                when(col(column).isNull(), 0)
                .otherwise(col(column))
            )
        elif column_type in ['float', 'double']:
            # Para columnas numéricas decimales
            df_cleaned = df_cleaned.withColumn(
                column,
                when(col(column).isNull(), 0.0)
                .otherwise(col(column))
            )
    
    print("✓ Columnas adicionales limpiadas")

print("\n6. ESTADO FINAL:")
print(f"Filas después de limpieza: {df_cleaned.count()}")
print("✓ Proceso de limpieza completado - NO se eliminaron columnas")

In [ ]:
# Verificación post-limpieza
print("=== VERIFICACIÓN POST-LIMPIEZA ===\n")

# Verificar que no hay valores nulos después de la limpieza
print("CONTEO DE VALORES NULOS DESPUÉS DE LIMPIEZA:")
null_counts_after = []

for column in df_cleaned.columns:
    null_count = df_cleaned.filter(col(column).isNull()).count()
    empty_count = df_cleaned.filter(col(column) == "").count()
    
    if null_count > 0 or empty_count > 0:
        null_counts_after.append({
            'column': column,
            'null_count': null_count,
            'empty_count': empty_count
        })

if null_counts_after:
    print("Columnas con valores problemáticos restantes:")
    for item in null_counts_after:
        print(f"  {item['column']}: {item['null_count']} nulos, {item['empty_count']} vacíos")
else:
    print("✓ ¡Excelente! No hay valores nulos o vacíos en el dataset limpio")

# Mostrar estadísticas de limpieza por columnas Top
print(f"\nESTADÍSTICAS DE LIMPIEZA - COLUMNAS TOP:")
sin_titular_counts = []
for column in top_columns[:5]:  # Mostrar solo primeras 5 para brevedad
    sin_titular_count = df_cleaned.filter(col(column) == "Sin titular").count()
    sin_titular_counts.append((column, sin_titular_count))
    
for column, count in sin_titular_counts:
    percentage = (count / df_cleaned.count()) * 100
    print(f"{column}: {count} 'Sin titular' ({percentage:.1f}%)")

print(f"\n✓ Dataset limpio listo para transformaciones")

## 6. Transformación de Datos (ETL)

Realizaremos transformaciones de datos incluyendo conversiones de tipos, creación de nuevas columnas y agregaciones usando funciones SQL de PySpark.

In [ ]:
print("=== TRANSFORMACIONES DE DATOS (ETL) ===\n")

# Crear DataFrame transformado
df_transformed = df_cleaned

# 1. Conversión de tipos de datos
print("1. CONVERSIÓN DE TIPOS DE DATOS:")

# Convertir Date a fecha si no está ya convertida
df_transformed = df_transformed.withColumn(
    "Date", 
    to_date(col("Date"), "yyyy-MM-dd")
)

# Asegurar que Label sea entero
df_transformed = df_transformed.withColumn(
    "Label",
    col("Label").cast(IntegerType())
)

print("✓ Date convertida a tipo fecha")
print("✓ Label convertida a tipo entero")

# 2. Crear nuevas columnas temporales
print("\n2. CREANDO NUEVAS COLUMNAS TEMPORALES:")

# Extraer año, mes, día de la semana
df_transformed = df_transformed.withColumn("Year", year(col("Date")))
df_transformed = df_transformed.withColumn("Month", month(col("Date")))
df_transformed = df_transformed.withColumn("DayOfMonth", dayofmonth(col("Date")))
df_transformed = df_transformed.withColumn("DayOfWeek", date_format(col("Date"), "EEEE"))
df_transformed = df_transformed.withColumn("WeekOfYear", weekofyear(col("Date")))

print("✓ Columnas Year, Month, DayOfMonth, DayOfWeek, WeekOfYear creadas")

# 3. Crear columnas de análisis de texto
print("\n3. CREANDO COLUMNAS DE ANÁLISIS DE TEXTO:")

# Contar titulares válidos (no "Sin titular")
top_columns = [col for col in df_transformed.columns if col.startswith('Top')]

# Crear expresión para contar titulares válidos
valid_headlines_expr = lit(0)
for column in top_columns:
    valid_headlines_expr = valid_headlines_expr + when(
        col(column) != "Sin titular", 1
    ).otherwise(0)

df_transformed = df_transformed.withColumn("ValidHeadlinesCount", valid_headlines_expr)

# Calcular longitud promedio de titulares
length_expr = lit(0)
count_expr = lit(0)

for column in top_columns:
    length_expr = length_expr + when(
        col(column) != "Sin titular", length(col(column))
    ).otherwise(0)
    count_expr = count_expr + when(
        col(column) != "Sin titular", 1
    ).otherwise(0)

df_transformed = df_transformed.withColumn(
    "AvgHeadlineLength",
    when(count_expr > 0, length_expr / count_expr).otherwise(0)
)

print("✓ Columna ValidHeadlinesCount creada")
print("✓ Columna AvgHeadlineLength creada")

# 4. Crear columnas categóricas
print("\n4. CREANDO COLUMNAS CATEGÓRICAS:")

# Clasificar sentimiento
df_transformed = df_transformed.withColumn(
    "SentimentCategory",
    when(col("Label") == 0, "Negative")
    .when(col("Label") == 1, "Positive")
    .otherwise("Unknown")
)

# Clasificar por época (trimestre)
df_transformed = df_transformed.withColumn(
    "Quarter",
    when(col("Month").isin([1, 2, 3]), "Q1")
    .when(col("Month").isin([4, 5, 6]), "Q2")
    .when(col("Month").isin([7, 8, 9]), "Q3")
    .when(col("Month").isin([10, 11, 12]), "Q4")
    .otherwise("Unknown")
)

# Identificar fin de semana
df_transformed = df_transformed.withColumn(
    "IsWeekend",
    when(col("DayOfWeek").isin(["Saturday", "Sunday"]), True)
    .otherwise(False)
)

print("✓ Columna SentimentCategory creada")
print("✓ Columna Quarter creada")
print("✓ Columna IsWeekend creada")

print(f"\n5. ESTADO FINAL DEL DATASET TRANSFORMADO:")
print(f"Filas: {df_transformed.count()}")
print(f"Columnas: {len(df_transformed.columns)}")
print("✓ Transformaciones completadas exitosamente")

In [ ]:
# Mostrar algunas estadísticas del dataset transformado
print("=== ESTADÍSTICAS DEL DATASET TRANSFORMADO ===\n")

# Registrar tabla temporal para usar SQL
df_transformed.createOrReplaceTempView("sentiment_data")

print("1. DISTRIBUCIÓN DE SENTIMIENTOS:")
sentiment_distribution = spark.sql("""
    SELECT SentimentCategory, COUNT(*) as Count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as Percentage
    FROM sentiment_data 
    GROUP BY SentimentCategory 
    ORDER BY Count DESC
""")
sentiment_distribution.show()

print("2. DISTRIBUCIÓN POR AÑOS:")
year_distribution = spark.sql("""
    SELECT Year, COUNT(*) as Count,
           AVG(ValidHeadlinesCount) as AvgValidHeadlines,
           AVG(AvgHeadlineLength) as AvgLength
    FROM sentiment_data 
    GROUP BY Year 
    ORDER BY Year
    LIMIT 10
""")
year_distribution.show()

print("3. DISTRIBUCIÓN POR TRIMESTRE:")
quarter_sentiment = spark.sql("""
    SELECT Quarter, SentimentCategory, COUNT(*) as Count
    FROM sentiment_data 
    GROUP BY Quarter, SentimentCategory 
    ORDER BY Quarter, SentimentCategory
""")
quarter_sentiment.show()

print("4. ANÁLISIS DE FIN DE SEMANA VS DÍAS LABORALES:")
weekend_analysis = spark.sql("""
    SELECT IsWeekend, SentimentCategory, COUNT(*) as Count,
           AVG(ValidHeadlinesCount) as AvgHeadlines
    FROM sentiment_data 
    GROUP BY IsWeekend, SentimentCategory 
    ORDER BY IsWeekend, SentimentCategory
""")
weekend_analysis.show()

print("✓ Análisis estadístico completado")

## 7. Visualización de Datos con PySpark y Matplotlib

Crearemos visualizaciones convirtiendo DataFrames de PySpark a Pandas para graficar con matplotlib, seaborn y plotly.

In [ ]:
print("=== CREANDO VISUALIZACIONES ===\n")

# Configurar estilo de matplotlib
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# 1. DISTRIBUCIÓN DE SENTIMIENTOS - Gráfico de Barras
print("1. Creando gráfico de distribución de sentimientos...")

# Convertir datos de PySpark a Pandas para visualización
sentiment_data = spark.sql("""
    SELECT SentimentCategory, COUNT(*) as Count
    FROM sentiment_data 
    GROUP BY SentimentCategory 
    ORDER BY Count DESC
""").toPandas()

# Crear figura con subplots
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Análisis de Sentimientos - Dashboard', fontsize=16, fontweight='bold')

# Gráfico de barras para sentimientos
bars = ax1.bar(sentiment_data['SentimentCategory'], sentiment_data['Count'], 
               color=['#ff6b6b', '#4ecdc4'], alpha=0.8)
ax1.set_title('Distribución de Sentimientos', fontweight='bold')
ax1.set_xlabel('Categoría de Sentimiento')
ax1.set_ylabel('Cantidad')

# Añadir etiquetas en las barras
for bar in bars:
    height = bar.get_height()
    ax1.annotate(f'{int(height)}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom')

# 2. DISTRIBUCIÓN POR AÑOS - Gráfico de Líneas
print("2. Creando gráfico de distribución por años...")

year_data = spark.sql("""
    SELECT Year, COUNT(*) as Count
    FROM sentiment_data 
    WHERE Year BETWEEN 2000 AND 2020
    GROUP BY Year 
    ORDER BY Year
""").toPandas()

ax2.plot(year_data['Year'], year_data['Count'], marker='o', linewidth=2, markersize=4)
ax2.set_title('Distribución de Noticias por Año', fontweight='bold')
ax2.set_xlabel('Año')
ax2.set_ylabel('Cantidad de Noticias')
ax2.grid(True, alpha=0.3)

# 3. SENTIMIENTOS POR TRIMESTRE - Gráfico de Barras Agrupadas
print("3. Creando gráfico de sentimientos por trimestre...")

quarter_data = spark.sql("""
    SELECT Quarter, SentimentCategory, COUNT(*) as Count
    FROM sentiment_data 
    GROUP BY Quarter, SentimentCategory 
    ORDER BY Quarter, SentimentCategory
""").toPandas()

# Pivotar datos para el gráfico
quarter_pivot = quarter_data.pivot(index='Quarter', columns='SentimentCategory', values='Count')
quarter_pivot.plot(kind='bar', ax=ax3, color=['#ff6b6b', '#4ecdc4'])
ax3.set_title('Sentimientos por Trimestre', fontweight='bold')
ax3.set_xlabel('Trimestre')
ax3.set_ylabel('Cantidad')
ax3.legend(title='Sentimiento')
ax3.tick_params(axis='x', rotation=0)

# 4. ANÁLISIS DE TITULARES VÁLIDOS - Histograma
print("4. Creando histograma de titulares válidos...")

headlines_data = spark.sql("""
    SELECT ValidHeadlinesCount
    FROM sentiment_data
    WHERE ValidHeadlinesCount > 0
""").toPandas()

ax4.hist(headlines_data['ValidHeadlinesCount'], bins=20, alpha=0.7, color='#45b7d1', edgecolor='black')
ax4.set_title('Distribución de Titulares Válidos por Día', fontweight='bold')
ax4.set_xlabel('Número de Titulares Válidos')
ax4.set_ylabel('Frecuencia')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Visualizaciones básicas completadas")

In [ ]:
# Visualizaciones avanzadas con Seaborn y Plotly
print("=== VISUALIZACIONES AVANZADAS ===\n")

# 5. HEATMAP DE SENTIMIENTOS POR MES Y AÑO
print("5. Creando heatmap de sentimientos por mes y año...")

monthly_sentiment = spark.sql("""
    SELECT Year, Month, 
           AVG(CASE WHEN Label = 1 THEN 1.0 ELSE 0.0 END) as PositivePct
    FROM sentiment_data 
    WHERE Year BETWEEN 2000 AND 2005  -- Limitamos para mejor visualización
    GROUP BY Year, Month 
    ORDER BY Year, Month
""").toPandas()

# Crear pivot para heatmap
monthly_pivot = monthly_sentiment.pivot(index='Month', columns='Year', values='PositivePct')

plt.figure(figsize=(12, 8))
sns.heatmap(monthly_pivot, annot=True, fmt='.2f', cmap='RdYlBu_r', 
            cbar_kws={'label': 'Porcentaje de Sentimiento Positivo'})
plt.title('Heatmap: Porcentaje de Sentimiento Positivo por Mes y Año', fontsize=14, fontweight='bold')
plt.xlabel('Año')
plt.ylabel('Mes')
plt.tight_layout()
plt.show()

# 6. GRÁFICO DE VIOLÍN - LONGITUD DE TITULARES POR SENTIMIENTO
print("6. Creando gráfico de violín para longitud de titulares...")

length_sentiment = spark.sql("""
    SELECT SentimentCategory, AvgHeadlineLength
    FROM sentiment_data 
    WHERE AvgHeadlineLength > 0 AND AvgHeadlineLength < 100  -- Filtrar valores extremos
""").toPandas()

plt.figure(figsize=(10, 6))
sns.violinplot(data=length_sentiment, x='SentimentCategory', y='AvgHeadlineLength', 
               palette=['#ff6b6b', '#4ecdc4'])
plt.title('Distribución de Longitud Promedio de Titulares por Sentimiento', 
          fontsize=14, fontweight='bold')
plt.xlabel('Categoría de Sentimiento')
plt.ylabel('Longitud Promedio de Titulares')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 7. GRÁFICO INTERACTIVO CON PLOTLY
print("7. Creando gráfico interactivo con Plotly...")

# Datos para gráfico temporal
temporal_data = spark.sql("""
    SELECT Date, SentimentCategory, COUNT(*) as Count
    FROM sentiment_data 
    WHERE Year BETWEEN 2000 AND 2002  -- Muestra para mejor rendimiento
    GROUP BY Date, SentimentCategory 
    ORDER BY Date
""").toPandas()

# Crear gráfico interactivo
fig_plotly = px.line(temporal_data, x='Date', y='Count', color='SentimentCategory',
                     title='Evolución Temporal de Sentimientos (2000-2002)',
                     labels={'Count': 'Cantidad de Noticias', 'Date': 'Fecha'})

fig_plotly.update_layout(
    width=1000,
    height=500,
    title_font_size=16,
    legend_title_text='Sentimiento'
)

fig_plotly.show()

print("✓ Visualizaciones avanzadas completadas")

In [ ]:
# Análisis adicional y gráficos de correlación
print("=== ANÁLISIS DE CORRELACIONES Y PATRONES ===\n")

# 8. ANÁLISIS DE FIN DE SEMANA VS DÍAS LABORALES
print("8. Creando análisis de fin de semana vs días laborales...")

weekend_sentiment = spark.sql("""
    SELECT 
        CASE WHEN IsWeekend THEN 'Fin de Semana' ELSE 'Día Laboral' END as DayType,
        SentimentCategory, 
        COUNT(*) as Count,
        AVG(ValidHeadlinesCount) as AvgHeadlines
    FROM sentiment_data 
    GROUP BY IsWeekend, SentimentCategory 
    ORDER BY IsWeekend, SentimentCategory
""").toPandas()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Gráfico de barras agrupadas para conteos
weekend_count = weekend_sentiment.pivot(index='DayType', columns='SentimentCategory', values='Count')
weekend_count.plot(kind='bar', ax=ax1, color=['#ff6b6b', '#4ecdc4'])
ax1.set_title('Distribución de Sentimientos: Fin de Semana vs Días Laborales', fontweight='bold')
ax1.set_xlabel('Tipo de Día')
ax1.set_ylabel('Cantidad de Noticias')
ax1.legend(title='Sentimiento')
ax1.tick_params(axis='x', rotation=45)

# Gráfico de promedio de titulares
weekend_avg = weekend_sentiment.pivot(index='DayType', columns='SentimentCategory', values='AvgHeadlines')
weekend_avg.plot(kind='bar', ax=ax2, color=['#ff9999', '#66cccc'])
ax2.set_title('Promedio de Titulares Válidos por Tipo de Día', fontweight='bold')
ax2.set_xlabel('Tipo de Día')
ax2.set_ylabel('Promedio de Titulares')
ax2.legend(title='Sentimiento')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# 9. TOP 5 AÑOS CON MÁS SENTIMIENTO POSITIVO
print("9. Analizando años con mayor sentimiento positivo...")

positive_years = spark.sql("""
    SELECT Year, 
           COUNT(CASE WHEN Label = 1 THEN 1 END) as PositiveCount,
           COUNT(*) as TotalCount,
           ROUND(COUNT(CASE WHEN Label = 1 THEN 1 END) * 100.0 / COUNT(*), 2) as PositivePct
    FROM sentiment_data 
    WHERE Year IS NOT NULL AND Year > 1990
    GROUP BY Year 
    ORDER BY PositivePct DESC
    LIMIT 10
""").toPandas()

plt.figure(figsize=(12, 6))
bars = plt.bar(positive_years['Year'].astype(str), positive_years['PositivePct'], 
               color='#4ecdc4', alpha=0.8)
plt.title('Top 10 Años con Mayor Porcentaje de Sentimiento Positivo', fontsize=14, fontweight='bold')
plt.xlabel('Año')
plt.ylabel('Porcentaje de Sentimiento Positivo (%)')
plt.xticks(rotation=45)

# Añadir etiquetas en las barras
for i, bar in enumerate(bars):
    height = bar.get_height()
    plt.annotate(f'{height}%',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=8)

plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("✓ Análisis de correlaciones y patrones completado")

## 8. Guardar Dataset Limpio y Transformado

Guardaremos el dataset limpio y transformado en varios formatos (CSV, Parquet) usando las operaciones de escritura de PySpark.

In [ ]:
print("=== GUARDANDO DATASET LIMPIO Y TRANSFORMADO ===\n")

# Crear directorio de salida si no existe
import os
output_dir = "Files/processed"
graphics_dir = "Graphics"

# Mostrar resumen final del dataset
print("1. RESUMEN FINAL DEL DATASET:")
print(f"   - Filas totales: {df_transformed.count()}")
print(f"   - Columnas totales: {len(df_transformed.columns())}")
print(f"   - Columnas nuevas creadas: Year, Month, DayOfMonth, DayOfWeek, WeekOfYear,")
print(f"     ValidHeadlinesCount, AvgHeadlineLength, SentimentCategory, Quarter, IsWeekend")

# Mostrar las primeras filas del dataset final
print("\n2. MUESTRA DEL DATASET TRANSFORMADO:")
df_transformed.select(
    "Date", "Label", "SentimentCategory", "Year", "Month", "Quarter", 
    "ValidHeadlinesCount", "AvgHeadlineLength", "IsWeekend"
).show(5)

# Guardar en formato CSV
print("\n3. GUARDANDO EN FORMATO CSV:")
try:
    # Coalescer para evitar múltiples archivos pequeños
    df_transformed.coalesce(1).write.mode("overwrite").option("header", "true").csv("Files/sentiment_data_transformed")
    print("✓ Dataset guardado en formato CSV en: Files/sentiment_data_transformed/")
except Exception as e:
    print(f"⚠ Error guardando CSV: {e}")

# Guardar en formato Parquet (más eficiente para big data)
print("\n4. GUARDANDO EN FORMATO PARQUET:")
try:
    df_transformed.write.mode("overwrite").parquet("Files/sentiment_data_parquet")
    print("✓ Dataset guardado en formato Parquet en: Files/sentiment_data_parquet/")
except Exception as e:
    print(f"⚠ Error guardando Parquet: {e}")

# Guardar estadísticas de resumen
print("\n5. CREANDO ARCHIVO DE ESTADÍSTICAS:")
try:
    # Crear resumen estadístico
    summary_stats = spark.sql("""
        SELECT 
            COUNT(*) as total_records,
            COUNT(DISTINCT Year) as unique_years,
            MIN(Date) as earliest_date,
            MAX(Date) as latest_date,
            AVG(ValidHeadlinesCount) as avg_headlines_per_day,
            AVG(AvgHeadlineLength) as avg_headline_length,
            SUM(CASE WHEN Label = 1 THEN 1 ELSE 0 END) as positive_sentiment_count,
            SUM(CASE WHEN Label = 0 THEN 1 ELSE 0 END) as negative_sentiment_count,
            ROUND(AVG(CASE WHEN Label = 1 THEN 1.0 ELSE 0.0 END) * 100, 2) as positive_sentiment_percentage
        FROM sentiment_data
    """)
    
    summary_stats.coalesce(1).write.mode("overwrite").option("header", "true").csv("Files/dataset_summary_statistics")
    print("✓ Estadísticas de resumen guardadas en: Files/dataset_summary_statistics/")
except Exception as e:
    print(f"⚠ Error guardando estadísticas: {e}")

# Guardar dataset solo con columnas principales (versión ligera)
print("\n6. CREANDO VERSIÓN LIGERA DEL DATASET:")
try:
    df_light = df_transformed.select(
        "Date", "Label", "SentimentCategory", "Year", "Month", "Quarter",
        "DayOfWeek", "ValidHeadlinesCount", "AvgHeadlineLength", "IsWeekend",
        "Top1", "Top2", "Top3", "Top4", "Top5"  # Solo primeros 5 titulares
    )
    
    df_light.coalesce(1).write.mode("overwrite").option("header", "true").csv("Files/sentiment_data_light")
    print("✓ Versión ligera guardada en: Files/sentiment_data_light/")
    print(f"   - Columnas reducidas de {len(df_transformed.columns)} a {len(df_light.columns)}")
except Exception as e:
    print(f"⚠ Error guardando versión ligera: {e}")

print(f"\n✓ ¡Proceso ETL completado exitosamente!")
print(f"  - Dataset original limpiado (sin eliminar columnas)")
print(f"  - Nuevas columnas agregadas para análisis")
print(f"  - Visualizaciones creadas")
print(f"  - Datasets guardados en múltiples formatos")

## 9. Limpieza y Cierre de Sesión Spark

Finalmente, limpiaremos los recursos y cerraremos la sesión de Spark apropiadamente.

In [ ]:
# Limpieza de recursos y cierre de Spark Session
print("=== LIMPIEZA Y CIERRE DE SESIÓN ===\n")

# Mostrar información de la sesión antes de cerrar
print("1. INFORMACIÓN DE LA SESIÓN SPARK:")
print(f"   - Aplicación: {spark.sparkContext.appName}")
print(f"   - Versión de Spark: {spark.version}")
print(f"   - Master: {spark.sparkContext.master}")
print(f"   - Spark UI: {spark.sparkContext.uiWebUrl}")

# Limpiar caché si se utilizó
print("\n2. LIMPIANDO CACHÉ:")
try:
    spark.catalog.clearCache()
    print("✓ Caché limpiado exitosamente")
except Exception as e:
    print(f"⚠ Error limpiando caché: {e}")

# Eliminar tablas temporales
print("\n3. ELIMINANDO TABLAS TEMPORALES:")
try:
    spark.catalog.dropTempView("sentiment_data")
    print("✓ Vista temporal 'sentiment_data' eliminada")
except Exception as e:
    print(f"⚠ No se pudo eliminar vista temporal: {e}")

# Mostrar resumen final del proceso ETL
print("\n" + "="*60)
print("                    RESUMEN DEL PROCESO ETL")
print("="*60)
print("✓ Librerías instaladas e importadas correctamente")
print("✓ Sesión de Spark inicializada exitosamente")
print("✓ Dataset cargado desde archivos CSV")
print("✓ Análisis exploratorio completado")
print("✓ Datos limpiados SIN eliminar columnas")
print("✓ Transformaciones ETL aplicadas")
print("✓ Nuevas columnas creadas para análisis")
print("✓ Visualizaciones generadas con matplotlib, seaborn y plotly")
print("✓ Dataset guardado en múltiples formatos (CSV, Parquet)")
print("✓ Estadísticas de resumen generadas")
print("="*60)

print(f"\n4. ARCHIVOS GENERADOS:")
print(f"   - Files/sentiment_data_transformed/ (CSV completo)")
print(f"   - Files/sentiment_data_parquet/ (Parquet)")
print(f"   - Files/dataset_summary_statistics/ (Estadísticas)")
print(f"   - Files/sentiment_data_light/ (Versión ligera)")

# Cerrar Spark Session
print(f"\n5. CERRANDO SESIÓN SPARK:")
try:
    spark.stop()
    print("✓ Sesión de Spark cerrada correctamente")
    print("✓ Recursos liberados")
except Exception as e:
    print(f"⚠ Error cerrando sesión: {e}")

print(f"\n🎉 ¡PROCESO ETL COMPLETADO EXITOSAMENTE! 🎉")
print(f"El notebook ha procesado los datos de análisis de sentimientos")
print(f"utilizando PySpark para limpieza, transformación y visualización.")

## 📋 Conclusiones y Próximos Pasos

### Resumen del Proceso ETL Realizado:

1. **✅ Extracción**: Se cargaron exitosamente los datasets de análisis de sentimientos
2. **✅ Transformación**: Se limpiaron los datos SIN eliminar columnas, se crearon nuevas variables
3. **✅ Carga**: Se guardaron los datos procesados en múltiples formatos

### Hallazgos Principales:
- Dataset con **6,087 registros** de noticias con análisis de sentimientos
- **27 columnas** originales mantenidas + **10 nuevas columnas** creadas
- Distribución de sentimientos entre positivos (1) y negativos (0)
- Análisis temporal desde el año 2000 en adelante
- Patrones identificados en fin de semana vs días laborales

### Nuevas Columnas Creadas:
- `Year`, `Month`, `DayOfMonth`, `DayOfWeek`, `WeekOfYear`: Variables temporales
- `ValidHeadlinesCount`: Cantidad de titulares válidos por día
- `AvgHeadlineLength`: Longitud promedio de titulares
- `SentimentCategory`: Categoría legible del sentimiento
- `Quarter`: Trimestre del año
- `IsWeekend`: Indicador de fin de semana

### Archivos Generados:
- **CSV completo**: `Files/sentiment_data_transformed/`
- **Parquet**: `Files/sentiment_data_parquet/` (formato optimizado)
- **Estadísticas**: `Files/dataset_summary_statistics/`
- **Versión ligera**: `Files/sentiment_data_light/`

### Próximos Pasos Sugeridos:
1. **Análisis de Machine Learning**: Usar los datos para entrenar modelos predictivos
2. **Análisis de texto**: NLP sobre los titulares para extraer temas
3. **Dashboard interactivo**: Crear un dashboard en tiempo real con Dash o Streamlit
4. **Análisis de series temporales**: Estudiar tendencias temporales más profundas

### Tecnologías Utilizadas:
- **PySpark**: Procesamiento distribuido de big data
- **Matplotlib/Seaborn**: Visualizaciones estáticas
- **Plotly**: Visualizaciones interactivas
- **Pandas**: Análisis de datos complementario